# Original dataset audit and group-aware partitioning

Sanitized source from the original data-audit notebook, restricted to the audit and split stage. Exact shipped split IDs are authoritative for FINAL V2. This optional stage recomputes groups and folds for inspection; it does not replace the published IDs. Run data preparation first. It reads the downloaded data under `work/` and writes recomputed split files under `work/audit_recomputed/`. No images, captions, or notebook outputs are bundled.


In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

base_dir = ((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "work")
df = pd.read_csv(base_dir / "dataset.csv")

print(df.shape)
print(df.head())
print(df["label"].value_counts())
print(df["platform"].value_counts())
print(df.isna().sum())

In [ ]:
pd.crosstab(df["platform"], df["label"], margins=True)

In [ ]:
duplicate_captions = df[df.duplicated(subset=["text_caption"], keep=False)]
print("Jumlah duplicated captions:", duplicate_captions.shape[0])
duplicate_captions.sort_values("text_caption").head(20)

In [ ]:
from pathlib import Path

base_dir = ((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "work")
df["image_exists"] = df["image_path"].apply(lambda p: (base_dir / p).exists())

print(df["image_exists"].value_counts())
df[df["image_exists"] == False].head()

In [ ]:
# Audit duplikasi caption

import re
import unicodedata
import pandas as pd

# Salin dataframe supaya data asli tidak ikut berubah.
audit_df = df.copy()

def normalize_caption(text):
    """
    Normalisasi untuk mendeteksi caption yang secara substansi sama:
    - Unicode dinormalisasi
    - huruf kecil
    - URL dihapus
    - tanda baca/simbol diubah menjadi spasi
    - spasi berulang dirapikan
    """
    text = "" if pd.isna(text) else str(text)

    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    # Hapus URL
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Sisakan huruf dan angka
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Rapikan whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


audit_df["normalized_caption"] = (
    audit_df["text_caption"]
    .apply(normalize_caption)
)

# --------------------------------------------------
# 1. Exact duplicate berdasarkan caption asli
# --------------------------------------------------

exact_duplicate_mask = audit_df.duplicated(
    subset=["text_caption"],
    keep=False
)

exact_duplicates = audit_df[exact_duplicate_mask].copy()

# --------------------------------------------------
# 2. Duplicate setelah caption dinormalisasi
# --------------------------------------------------

normalized_duplicate_mask = audit_df.duplicated(
    subset=["normalized_caption"],
    keep=False
)

normalized_duplicates = audit_df[
    normalized_duplicate_mask
].copy()

# Abaikan normalized caption kosong
normalized_duplicates = normalized_duplicates[
    normalized_duplicates["normalized_caption"] != ""
]

# --------------------------------------------------
# 3. Caption sama tetapi label berbeda
# --------------------------------------------------

label_per_caption = (
    audit_df[
        audit_df["normalized_caption"] != ""
    ]
    .groupby("normalized_caption")["label"]
    .nunique()
)

conflicting_caption_keys = label_per_caption[
    label_per_caption > 1
].index

conflicting_labels = audit_df[
    audit_df["normalized_caption"].isin(
        conflicting_caption_keys
    )
].sort_values(
    ["normalized_caption", "label", "platform"]
)

# --------------------------------------------------
# 4. Caption sama muncul di platform berbeda
# --------------------------------------------------

platform_per_caption = (
    audit_df[
        audit_df["normalized_caption"] != ""
    ]
    .groupby("normalized_caption")["platform"]
    .nunique()
)

cross_platform_keys = platform_per_caption[
    platform_per_caption > 1
].index

cross_platform_duplicates = audit_df[
    audit_df["normalized_caption"].isin(
        cross_platform_keys
    )
].sort_values(
    ["normalized_caption", "platform"]
)

# --------------------------------------------------
# Ringkasan
# --------------------------------------------------

summary = pd.Series({
    "Total rows": len(audit_df),

    "Rows in exact duplicate groups":
        len(exact_duplicates),

    "Number of exact duplicate groups":
        exact_duplicates["text_caption"].nunique(),

    "Rows in normalized duplicate groups":
        len(normalized_duplicates),

    "Number of normalized duplicate groups":
        normalized_duplicates[
            "normalized_caption"
        ].nunique(),

    "Conflicting-label groups":
        len(conflicting_caption_keys),

    "Rows in conflicting-label groups":
        len(conflicting_labels),

    "Cross-platform duplicate groups":
        len(cross_platform_keys),

    "Rows in cross-platform duplicate groups":
        len(cross_platform_duplicates),
})

print("=== DUPLICATE AUDIT SUMMARY ===")
print(summary)

print("\n=== CONFLICTING LABEL EXAMPLES ===")
display(
    conflicting_labels[
        [
            "id",
            "text_caption",
            "normalized_caption",
            "platform",
            "label"
        ]
    ].head(20)
)

print("\n=== NORMALIZED DUPLICATE EXAMPLES ===")
display(
    normalized_duplicates[
        [
            "id",
            "text_caption",
            "normalized_caption",
            "platform",
            "label"
        ]
    ]
    .sort_values("normalized_caption")
    .head(30)
)

In [ ]:
# Audit duplikasi Image

from pathlib import Path
import hashlib
import pandas as pd

# Pastikan base_dir sudah menunjuk ke folder utama dataset.
# Contoh:
# base_dir = Path("indonesian-gambling-multimodal-dataset")

def calculate_file_hash(relative_path):
    """
    Menghasilkan SHA-256 dari isi file gambar.
    Gambar dengan hash sama berarti file-nya identik.
    """
    image_file = base_dir / relative_path

    if not image_file.exists():
        return None

    sha256 = hashlib.sha256()

    with open(image_file, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


image_audit_df = audit_df.copy()

image_audit_df["image_sha256"] = (
    image_audit_df["image_path"]
    .apply(calculate_file_hash)
)

# Ambil semua baris yang mempunyai hash gambar berulang
exact_image_duplicate_mask = image_audit_df.duplicated(
    subset=["image_sha256"],
    keep=False
)

exact_image_duplicates = (
    image_audit_df[
        exact_image_duplicate_mask
        & image_audit_df["image_sha256"].notna()
    ]
    .sort_values(["image_sha256", "label", "platform"])
    .copy()
)

# Cek gambar identik tetapi label berbeda
labels_per_image = (
    image_audit_df[
        image_audit_df["image_sha256"].notna()
    ]
    .groupby("image_sha256")["label"]
    .nunique()
)

conflicting_image_hashes = labels_per_image[
    labels_per_image > 1
].index

conflicting_image_labels = (
    image_audit_df[
        image_audit_df["image_sha256"].isin(
            conflicting_image_hashes
        )
    ]
    .sort_values(["image_sha256", "label", "platform"])
)

summary_image = pd.Series({
    "Total rows": len(image_audit_df),

    "Images successfully hashed":
        image_audit_df["image_sha256"].notna().sum(),

    "Rows in exact duplicate image groups":
        len(exact_image_duplicates),

    "Number of exact duplicate image groups":
        exact_image_duplicates["image_sha256"].nunique(),

    "Conflicting-label image groups":
        len(conflicting_image_hashes),

    "Rows in conflicting-label image groups":
        len(conflicting_image_labels),
})

print("=== EXACT IMAGE DUPLICATE AUDIT ===")
print(summary_image)

print("\n=== EXACT DUPLICATE IMAGE EXAMPLES ===")
display(
    exact_image_duplicates[
        [
            "id",
            "text_caption",
            "image_path",
            "platform",
            "label",
            "image_sha256"
        ]
    ].head(30)
)

print("\n=== CONFLICTING IMAGE LABEL EXAMPLES ===")
display(
    conflicting_image_labels[
        [
            "id",
            "text_caption",
            "image_path",
            "platform",
            "label",
            "image_sha256"
        ]
    ].head(30)
)


In [ ]:
# Membuat leakage group gabungan berdasarkan:
# 1. normalized caption yang sama
# 2. exact image hash yang sama

group_df = image_audit_df.copy().reset_index(drop=True)

class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        root_x = self.find(x)
        root_y = self.find(y)

        if root_x == root_y:
            return

        if self.rank[root_x] < self.rank[root_y]:
            self.parent[root_x] = root_y
        elif self.rank[root_x] > self.rank[root_y]:
            self.parent[root_y] = root_x
        else:
            self.parent[root_y] = root_x
            self.rank[root_x] += 1


uf = UnionFind(len(group_df))

# --------------------------------------------------
# Gabungkan baris dengan normalized caption yang sama
# --------------------------------------------------

for _, indices in (
    group_df[
        group_df["normalized_caption"].notna()
        & (group_df["normalized_caption"] != "")
    ]
    .groupby("normalized_caption")
    .groups
    .items()
):
    indices = list(indices)
    first_index = indices[0]

    for other_index in indices[1:]:
        uf.union(first_index, other_index)

# --------------------------------------------------
# Gabungkan baris dengan image hash yang sama
# --------------------------------------------------

for _, indices in (
    group_df[
        group_df["image_sha256"].notna()
    ]
    .groupby("image_sha256")
    .groups
    .items()
):
    indices = list(indices)
    first_index = indices[0]

    for other_index in indices[1:]:
        uf.union(first_index, other_index)

# --------------------------------------------------
# Buat leakage group ID
# --------------------------------------------------

roots = [uf.find(index) for index in range(len(group_df))]

root_to_group_id = {
    root: group_id
    for group_id, root in enumerate(sorted(set(roots)))
}

group_df["leakage_group_id"] = [
    root_to_group_id[root]
    for root in roots
]

# --------------------------------------------------
# Ringkasan ukuran group
# --------------------------------------------------

group_sizes = (
    group_df
    .groupby("leakage_group_id")
    .size()
    .sort_values(ascending=False)
)

# Cek group yang mempunyai lebih dari satu label
labels_per_group = (
    group_df
    .groupby("leakage_group_id")["label"]
    .nunique()
)

mixed_label_group_ids = labels_per_group[
    labels_per_group > 1
].index

mixed_label_groups = (
    group_df[
        group_df["leakage_group_id"].isin(
            mixed_label_group_ids
        )
    ]
    .sort_values(
        ["leakage_group_id", "label", "platform"]
    )
)

summary_groups = pd.Series({
    "Total rows":
        len(group_df),

    "Total leakage groups":
        group_df["leakage_group_id"].nunique(),

    "Groups containing more than one row":
        (group_sizes > 1).sum(),

    "Rows inside multi-row groups":
        group_sizes[group_sizes > 1].sum(),

    "Largest group size":
        group_sizes.max(),

    "Mixed-label leakage groups":
        len(mixed_label_group_ids),

    "Rows inside mixed-label groups":
        len(mixed_label_groups),
})

print("=== UNIFIED LEAKAGE GROUP SUMMARY ===")
print(summary_groups.to_string())

print("\n=== LARGEST LEAKAGE GROUPS ===")
display(group_sizes.head(15).rename("group_size").to_frame())

print("\n=== MIXED-LABEL GROUPS ===")
display(
    mixed_label_groups[
        [
            "id",
            "text_caption",
            "normalized_caption",
            "image_path",
            "platform",
            "label",
            "leakage_group_id"
        ]
    ]
)

In [ ]:
# Buat group-aware split 70/15/15

from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd

split_df = group_df.copy()

# Stratifikasi berdasarkan kombinasi label dan platform.
# Contoh strata: 0_Facebook, 1_TikTok, 1_X
split_df["stratify_key"] = (
    split_df["label"].astype(int).astype(str)
    + "_"
    + split_df["platform"].astype(str)
)

# 20 folds berarti setiap fold kira-kira 5%.
# 14 folds train = 70%
# 3 folds validation = 15%
# 3 folds test = 15%
sgkf = StratifiedGroupKFold(
    n_splits=20,
    shuffle=True,
    random_state=42
)

split_df["fold"] = -1

X_dummy = split_df[["id"]]
y = split_df["stratify_key"]
groups = split_df["leakage_group_id"]

for fold_number, (_, fold_indices) in enumerate(
    sgkf.split(X_dummy, y, groups)
):
    split_df.loc[fold_indices, "fold"] = fold_number

# Tetapkan split berdasarkan fold
split_df["split"] = "train"
split_df.loc[split_df["fold"].isin([14, 15, 16]), "split"] = "validation"
split_df.loc[split_df["fold"].isin([17, 18, 19]), "split"] = "test"

# --------------------------------------------------
# Pemeriksaan leakage
# --------------------------------------------------

splits_per_group = (
    split_df
    .groupby("leakage_group_id")["split"]
    .nunique()
)

leaking_groups = splits_per_group[
    splits_per_group > 1
]

# --------------------------------------------------
# Ringkasan ukuran split
# --------------------------------------------------

split_summary = (
    split_df["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .to_frame("rows")
)

split_summary["percentage"] = (
    split_summary["rows"]
    / len(split_df)
    * 100
).round(2)

print("=== SPLIT SUMMARY ===")
display(split_summary)

print(
    "Groups appearing in more than one split:",
    len(leaking_groups)
)

# --------------------------------------------------
# Distribusi label per split
# --------------------------------------------------

print("\n=== LABEL DISTRIBUTION ===")

label_distribution = pd.crosstab(
    split_df["split"],
    split_df["label"],
    margins=True
)

display(label_distribution)

# --------------------------------------------------
# Distribusi platform dan label per split
# --------------------------------------------------

print("\n=== PLATFORM × LABEL DISTRIBUTION ===")

platform_label_distribution = pd.crosstab(
    index=split_df["split"],
    columns=[
        split_df["platform"],
        split_df["label"]
    ]
)

display(platform_label_distribution)



In [ ]:
# Bekukan split
from pathlib import Path
import hashlib

split_dir = base_dir / "audit_recomputed" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

columns_to_save = [
    "id",
    "text_caption",
    "image_path",
    "platform",
    "label",
    "normalized_caption",
    "image_sha256",
    "leakage_group_id",
    "fold",
    "split",
]

# Simpan file master dan masing-masing split
split_df[columns_to_save].to_csv(
    split_dir / "split_manifest.csv",
    index=False
)

for split_name in ["train", "validation", "test"]:
    split_df.loc[
        split_df["split"] == split_name,
        columns_to_save
    ].to_csv(
        split_dir / f"{split_name}.csv",
        index=False
    )

# Buat checksum agar kita tahu file split tidak berubah
manifest_path = split_dir / "split_manifest.csv"

with open(manifest_path, "rb") as file:
    checksum = hashlib.sha256(file.read()).hexdigest()

print("Split manifest:", manifest_path)
print("SHA-256:", checksum)

print("\nFiles created:")
for path in sorted(split_dir.glob("*.csv")):
    print(f"- {path}: {path.stat().st_size:,} bytes")